# DYS-HJ for Sparse-Group LASSO

Compares Davis–Yin three-operator splitting using analytical proximals (soft-thresholding for ℓ₁ + group shrinkage for the overlapping-group penalty) against DYS using HJ-Prox.  Reproduces a panel of Figure 2.

## Setup


In [ ]:
# ============================================================================
# CHUNK 1: SETUP - Algorithms, Helper Functions, and Definitions
# ============================================================================

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.linalg import norm
from hj_prox import hj_prox
import time

EPS = 1e-5

# Plotting configuration
plt.rcParams.update({'font.size': 20})


# ============================================================================
# Helper Functions
# ============================================================================

def soft_threshold(x, threshold):
    """Proximal operator for L1 norm (soft thresholding)."""
    return np.sign(x) * np.maximum(np.abs(x) - threshold, 0)


def prox_group_lasso(x, groups, threshold):
    """Proximal operator for group L2 norm."""
    x_prox = x.copy()
    for start, end in groups:
        group = x[start:end]
        group_norm = norm(group, 2)
        if group_norm > 0:
            scale = np.maximum(1 - threshold * np.sqrt(end - start) / group_norm, 0)
            x_prox[start:end] = scale * group
        else:
            x_prox[start:end] = 0
    return x_prox


def gradient_f_numpy(X, y, beta):
    """Gradient of the smooth term f(β) = (1/2)||y - Xβ||²."""
    return X.T @ (X @ beta - y)


def objective_numpy(X, y, beta, groups, lambda_1, lambda_2):
    """Compute the full objective function value."""
    residual = y - X @ beta
    smooth_term = 0.5 * np.sum(residual ** 2)
    l1_term = lambda_1 * np.sum(np.abs(beta))
    
    group_term = 0
    for start, end in groups:
        group_term += lambda_2 * np.sqrt(end - start) * norm(beta[start:end], 2)
    
    return smooth_term + l1_term + group_term


def l1_penalty_batch(beta_batch, lambda_1):
    """Compute L1 penalty for batch of coefficient vectors.
    Args:
        beta_batch: shape (n_samples, n_features)
        lambda_1: L1 penalty parameter
    Returns:
        penalties: shape (n_samples,)
    """
    return lambda_1 * torch.abs(beta_batch).sum(dim=1)


def group_penalty_batch(beta_batch, groups, lambda_2):
    """Compute group L2 penalty for batch of coefficient vectors.
    Args:
        beta_batch: shape (n_samples, n_features)
        groups: list of (start, end) tuples
        lambda_2: group penalty parameter
    Returns:
        penalties: shape (n_samples,)
    """
    penalties = torch.zeros(beta_batch.shape[0], device=beta_batch.device)
    for start, end in groups:
        group_size = end - start
        # Compute L2 norm for each sample's group
        group_norms = torch.norm(beta_batch[:, start:end], p=2, dim=1)
        penalties += lambda_2 * np.sqrt(group_size) * group_norms
    return penalties


def compute_gradient(beta, X, y):
    """Compute gradient of smooth term: ∇f(β) = X^T(Xβ - y)"""
    return X.T @ (X @ beta - y)


def total_objective(beta, X, y, groups, lambda_1, lambda_2):
    """Compute total objective value."""
    residual = y - X @ beta
    smooth_part = 0.5 * (residual**2).sum()
    l1_part = lambda_1 * torch.abs(beta).sum()
    
    group_part = 0
    for start, end in groups:
        group_size = end - start
        group_norm = torch.norm(beta[start:end], p=2)
        group_part += lambda_2 * np.sqrt(group_size) * group_norm
    
    return smooth_part + l1_part + group_part


# ============================================================================
# Algorithm 1: Davis-Yin with Analytical Proximal Operators
# ============================================================================

def davis_yin_standard(X, y, groups, lambda_1, lambda_2, gamma=None, max_iter=1000, tol=1e-25):
    """Standard Davis-Yin with exact proximal operators."""
    n_samples, n_features = X.shape
    
    # Compute Lipschitz constant if gamma not provided
    if gamma is None:
        L = norm(X.T @ X, 2)
        gamma = 1.0 / L
    
    # Initialize variables
    beta = np.zeros(n_features)
    z = beta.copy()
    
    objectives = []
    
    print("Starting Standard Davis-Yin Algorithm...")
    print(f"λ₁ = {lambda_1}, λ₂ = {lambda_2}, γ = {gamma}")
    print("-" * 70)
    
    for iteration in range(max_iter):
        t_start = time.time()
        beta_old = beta.copy()
        
        # Davis-Yin steps
        # Step 1: w = prox_{γh}(z)
        w = prox_group_lasso(z, groups, gamma * lambda_2)
        
        # Step 2: beta = prox_{γg}(2w - z - γ∇f(w))
        grad = gradient_f_numpy(X, y, w)
        temp = 2 * w - z - gamma * grad
        beta = soft_threshold(temp, gamma * lambda_1)
        
        # Step 3: z = z + (beta - w)
        z = z + (beta - w)
        
        # Compute objective
        obj_val = objective_numpy(X, y, beta, groups, lambda_1, lambda_2)
        objectives.append(obj_val)
        
        t_iter = time.time() - t_start
        
        # Check convergence
        residual = norm(beta - beta_old) / (norm(beta_old) + 1e-10)
        
        if iteration % 5 == 0:
            print(f"Iteration {iteration:4d}: Objective = {obj_val:.2f}, "
                  f"||Δβ|| = {residual:.2e}, time = {t_iter:.3f}s")
        
        if residual < tol:
            print(f"\nConverged at iteration {iteration}")
            break
    
    return beta, objectives


# ============================================================================
# Algorithm 2: Davis-Yin with HJ-Prox
# ============================================================================

def davis_yin_hjprox(X, y, groups, lambda_1=0.1, lambda_2=0.1, 
                     gamma=1.0, gamma_min=1e-4, gamma_decay=0.99,
                     max_iter=1000, tol=1e-15,
                     num_samples_l1=100, num_samples_group=100,
                     delta_l1=1e-1, delta_group=1e-1,
                     beta_init=None, verbose=True, device='cpu'):
    """
    Davis-Yin algorithm using HJ-Prox for non-smooth terms with adaptive step size.
    
    Minimizes: f(β) + g(β) + h(β)
    where:
        f(β) = 0.5||y - Xβ||²  (smooth, use gradient)
        g(β) = λ₁||β||₁        (non-smooth, use HJ-Prox)
        h(β) = λ₂∑√pₘ||β^(m)||₂ (non-smooth, use HJ-Prox)
        
    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        Design matrix
    y : array-like, shape (n_samples,)
        Response vector
    groups : list of tuples
        List of (start, end) tuples defining groups
    lambda_1 : float
        L1 penalty parameter
    lambda_2 : float
        Group penalty parameter
    gamma : float
        Initial step size parameter
    gamma_min : float
        Minimum allowed value for gamma
    gamma_decay : float
        Factor to multiply gamma by when objective doesn't decrease
    max_iter : int
        Maximum number of iterations
    tol : float
        Convergence tolerance
    num_samples_l1 : int
        Number of samples for L1 HJ-Prox
    num_samples_group : int
        Number of samples for group HJ-Prox
    delta_l1 : float
        Smoothing parameter for L1 HJ-Prox
    delta_group : float
        Smoothing parameter for group HJ-Prox
    beta_init : array-like, optional
        Initial value for beta
    verbose : bool
        Whether to print progress
    device : str
        Device to use ('cpu' or 'cuda')
        
    Returns:
    --------
    beta_final : numpy array
        Optimized coefficients
    objectives : list
        Objective values at each iteration
    """
    # Convert to torch tensors
    X = torch.tensor(X, dtype=torch.float32, device=device)
    y = torch.tensor(y, dtype=torch.float32, device=device)
    
    n_samples, n_features = X.shape
    
    # Initialize beta
    if beta_init is None:
        beta = torch.zeros(n_features, device=device)
    else:
        if isinstance(beta_init, torch.Tensor):
            beta = beta_init.to(device=device, dtype=torch.float32)
        else:
            beta = torch.tensor(beta_init, dtype=torch.float32, device=device)
        
        if beta.shape[0] != n_features:
            raise ValueError(f"beta_init must have shape ({n_features},), but got shape {beta.shape}")
    
    z = beta.clone()
    
    objectives = []
    
    # Define penalty functions for HJ-Prox
    def l1_func(beta_batch):
        return l1_penalty_batch(beta_batch, lambda_1)
    
    def group_func(beta_batch):
        return group_penalty_batch(beta_batch, groups, lambda_2)
    
    # Initialize objective value
    prev_obj = total_objective(beta, X, y, groups, lambda_1, lambda_2).item()
    
    if verbose:
        print("Starting Davis-Yin with HJ-Prox (adaptive step size)...")
        print(f"λ₁ = {lambda_1}, λ₂ = {lambda_2}")
        print(f"Initial γ = {gamma}, γ_min = {gamma_min}, γ_decay = {gamma_decay}")
        if beta_init is not None:
            print(f"Using custom initial beta with ||β₀|| = {torch.norm(beta):.4f}")
        else:
            print("Starting from zero initialization")
        print("-" * 70)
    
    for iteration in range(max_iter):
        t_start = time.time()
        beta_old = beta.clone()
        
        # Compute delta with annealing schedule
        delta = 1500000 / (iteration + 1)**(2 + EPS)
        
        # Step 1: w = prox_{γh}(z) - proximal of group penalty
        z_flat = z.view(-1, 1)
        w_flat, ls_iters_group = hj_prox(
            z_flat, gamma, group_func,
            delta=delta, num_samples=num_samples_group,
            alpha=1.0
        )
        w = w_flat.view(n_features)
        
        # Step 2: beta = prox_{γg}(2w - z - γ∇f(w)) - proximal of L1 penalty
        grad_w = compute_gradient(w, X, y)
        v = 2 * w - z - gamma * grad_w
        v_flat = v.view(-1, 1)
        
        beta_flat, ls_iters_l1 = hj_prox(
            v_flat, gamma, l1_func,
            delta=delta, num_samples=num_samples_l1,
            alpha=1.0
        )
        beta = beta_flat.view(n_features)
        
        # Step 3: z = z + (beta - w)
        z = z + (beta - w)
        
        # Compute objective
        obj_val = total_objective(beta, X, y, groups, lambda_1, lambda_2)
        current_obj = obj_val.item()
        objectives.append(current_obj)
        
        # Track time
        t_iter = time.time() - t_start
        
        # Compute convergence metric
        diff = torch.norm(beta - beta_old).item()
        rel_diff = diff / (torch.norm(beta_old).item() + 1e-10)
        
        if verbose and iteration % 10 == 0:
            print(f"Iteration {iteration:4d}: Objective = {current_obj:.6f}, "
                  f"||Δβ|| = {diff:.2e}, γ = {gamma:.2e}, "
                  f"time = {t_iter:.3f}s, ls_iters = {ls_iters_l1 + ls_iters_group}")
        
        # Adaptive step size: decrease gamma if objective increased
        if current_obj > prev_obj * 1.01 and gamma > gamma_min:
            old_gamma = gamma
            gamma = max(gamma * 1, gamma_min)
            if verbose and iteration % 10 != 0:
                print(f"  -> Objective increased, reducing γ from {old_gamma:.2e} to {gamma:.2e}")
        
        prev_obj = current_obj
        
        # Check convergence
        if rel_diff < tol:
            if verbose:
                print(f"\nConverged at iteration {iteration}")
                print(f"Final objective: {current_obj:.6f}")
                print(f"Final ||β|| = {torch.norm(beta).item():.4f}")
                print(f"Non-zero coefficients: {(torch.abs(beta) > 1e-6).sum().item()}")
            break
    
    if iteration == max_iter - 1 and verbose:
        print(f"\nReached maximum iterations ({max_iter})")
        print(f"Final objective: {current_obj:.6f}")
    
    # Convert back to numpy
    beta_final = beta.cpu().numpy()
    
    return beta_final, objectives


print("✓ All algorithms and helper functions loaded successfully")


## Problem definition


In [ ]:
# ============================================================================
# CHUNK 2: DATA GENERATION
# ============================================================================

def generate_sparse_group_data(n_samples=500):
    """Generate dataset suitable for Sparse Group LASSO."""
    np.random.seed(42)
    
    group_sizes = [10, 10, 10, 10, 10, 10]
    n_features = sum(group_sizes)
    
    # Generate correlated features within groups
    X = np.random.randn(n_samples, n_features)
    feature_idx = 0
    for g, size in enumerate(group_sizes):
        group_correlation = 0.75
        for i in range(1, size):
            X[:, feature_idx + i] = (group_correlation * X[:, feature_idx] + 
                                     np.sqrt(1 - group_correlation**2) * X[:, feature_idx + i])
        feature_idx += size
    
    # Create true coefficients with group structure
    beta_true = np.zeros(n_features)
    # Group 1: features 2 and 5
    beta_true[2] = 1
    beta_true[5] = 1
    # Group 3: features 23, 26, and 29
    beta_true[23] = 1
    beta_true[26] = 1
    beta_true[29] = 1
    # Group 5: feature 45
    beta_true[45] = 1
    # Group 6: features 53, 54, 55
    beta_true[53] = 1
    beta_true[54] = 1
    beta_true[55] = 1
    
    # Generate response
    y = X @ beta_true + 0.25 * np.random.randn(n_samples)
    
    # Create group indices
    groups = []
    start = 0
    for size in group_sizes:
        groups.append((start, start + size))
        start += size
    
    group_names = ['group 1', 'group 2', 'group 3', 'group 4', 'group 5', 'group 6']
    
    return X, y, beta_true, groups, group_names


print("\n" + "="*70)
print("COMPARISON: STANDARD DAVIS-YIN vs HJ-PROX DAVIS-YIN")
print("="*70)

# Generate data
n = 300
X, y, beta_true, groups, group_names = generate_sparse_group_data(n)

print(f"\n✓ Dataset: {X.shape[0]} samples, {X.shape[1]} features in {len(groups)} groups")
print(f"✓ True model has {np.sum(beta_true != 0)} non-zero coefficients")

# Set parameters
lambda_1 = 0.15
lambda_2 = 0.01
L = norm(X.T @ X, 2)
gamma = 1.0 / L

print(f"✓ Lipschitz constant L = {L:.2f}")
print(f"✓ Step size γ = {gamma:.6f}")
print(f"✓ Penalty parameters: λ1={lambda_1}, λ2={lambda_2}")

## Algorithm 1 — DYS with analytical proximals


In [ ]:
# ============================================================================
# CHUNK 3: RUN ALGORITHM 1 - Analytical Davis-Yin
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 1: Standard Davis-Yin (Exact Proximals)...")
print("="*70)

start_time = time.time()

beta_Analytical, obj_Analytical = davis_yin_standard(
    X, y, groups, lambda_1*n, lambda_2*n, gamma=gamma*0.0072, max_iter=10000
)

elapsed_time = time.time() - start_time

print(f"\n✓ DYS-Analytical completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(obj_Analytical)} iterations")
print(f"  - Final objective: {obj_Analytical[-1]:.2f}")


## Algorithm 2 — DYS with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 4: RUN ALGORITHM 2 - Davis-Yin with HJ-Prox
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 2: Davis-Yin with HJ-Prox...")
print("="*70)

start_time = time.time()

beta_HJ, obj_HJ = davis_yin_hjprox(
    X, y, groups, lambda_1*n, lambda_2*n, gamma=gamma*0.0072, max_iter=10000,
    num_samples_l1=10000, num_samples_group=10000,
    delta_l1=.15, delta_group=.1, gamma_decay=1, gamma_min=gamma*0.005, verbose=True
)

elapsed_time = time.time() - start_time

print(f"\n✓ DYS-HJ completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(obj_HJ)} iterations")
print(f"  - Final objective: {obj_HJ[-1]:.6f}")

## Comparison: DYS vs DYS-HJ


In [ ]:
import os
os.makedirs('figures', exist_ok=True)
# ============================================================================
# CHUNK 5: GENERATE FIGURES
# ============================================================================

print("\n" + "="*70)
print("Generating figures...")
print("="*70)

# Get x positions
x_pos = np.arange(len(beta_true))

# --- Figure 1: Ground Truth ---
plt.figure(figsize=(11, 10))
plt.plot(x_pos, beta_true, 'o', markersize=10,
         markerfacecolor='none', markeredgecolor='black',
         markeredgewidth=3)
# Add vertical separators
for start, end in groups[:-1]:
    plt.axvline(x=end-0.5, color='gray', linestyle='--', alpha=0.3)
plt.ylabel('Coefficient Value', fontsize=40)
plt.xlabel('Coefficients', fontsize=40)
plt.title('Ground Truth', fontsize=40)
plt.grid(True, alpha=0.3)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/sparse_group_lasso_ground_truth.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 2: HJ-DYS Solution ---
plt.figure(figsize=(11, 10))
plt.plot(x_pos, beta_HJ, 's', markersize=8, color='blue')
# Add vertical separators
for start, end in groups[:-1]:
    plt.axvline(x=end-0.5, color='gray', linestyle='--', alpha=0.3)
plt.ylabel('Coefficient Value', fontsize=40)
plt.xlabel('Coefficient Index', fontsize=40)
plt.title('DYS-HJ', fontsize=40)
plt.grid(True, alpha=0.3)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/sparse_group_lasso_dys_hj.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 3: DYS Solution ---
plt.figure(figsize=(11, 10))
plt.plot(x_pos, beta_Analytical, '^', markersize=8, color='red')
# Add vertical separators
for start, end in groups[:-1]:
    plt.axvline(x=end-0.5, color='gray', linestyle='--', alpha=0.3)
plt.ylabel('Coefficient Value', fontsize=40)
plt.xlabel('Coefficients Index', fontsize=40)
plt.title('DYS', fontsize=40)
plt.grid(True, alpha=0.3)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/sparse_group_lasso_dys.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 4: Objective Function Convergence ---
plt.figure(figsize=(11, 10))
plt.semilogy(obj_HJ, '-', linewidth=3,
             label=f'DYS-HJ: {obj_HJ[-1]:.3f}')
plt.semilogy(obj_Analytical, '--', linewidth=3,
             label=f'DYS: {obj_Analytical[-1]:.3f}')
plt.ylabel('Objective Value (log scale)', fontsize=40)
plt.xlabel('Iteration', fontsize=40)
plt.title('DYS Convergence', fontsize=40)
plt.legend(fontsize=40, loc='upper left')
plt.grid(True, alpha=0.3, which='both')
tick_positions = np.arange(0, 10000 + 1, 2500)
plt.gca().set_xticks(tick_positions)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/sparse_group_lasso_objectives.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: sparse_group_lasso_ground_truth.pdf, sparse_group_lasso_dys_hj.pdf, sparse_group_lasso_dys.pdf, sparse_group_lasso_objectives.pdf")
print("\n" + "="*70)
print("✓ ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
print("="*70)